# Cybersecurity Dataset EDA and Readiness Audit

This notebook builds a practical understanding of the locally available source artifacts and prepared trajectories. It is memory-bounded by default, preserves dataset boundaries, and does not reinterpret attack classes as stages.

Run `uv sync --extra eda --extra pcap` and select the project `.venv` kernel.

## 1. Setup and safety limits

Increase the limits only after checking file sizes. Sampling is appropriate for schema and distribution exploration; final audit counts must use streaming/full preprocessing.

In [1]:
from __future__ import annotations

import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

RAW_ROW_LIMIT = 250_000
PCAP_PACKET_LIMIT = 50_000
RANDOM_SEED = 42


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the cyberworld repository")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
TRAJECTORY_ROOT = DATA_ROOT / "processed" / "trajectories"
print({"repo": str(ROOT), "raw": str(RAW_ROOT), "trajectories": str(TRAJECTORY_ROOT)})

{'repo': '/Users/shreyansh/sih-internal-ps2', 'raw': '/Users/shreyansh/sih-internal-ps2/data/raw', 'trajectories': '/Users/shreyansh/sih-internal-ps2/data/processed/trajectories'}


## 2. Artifact inventory

This answers what is actually present—not what a dataset web page says should exist.

In [2]:
def human_bytes(size: int) -> str:
    value = float(size)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:.2f} {unit}"
        value /= 1024
    return f"{value:.2f} TiB"


records = []
if RAW_ROOT.exists():
    for path in sorted(p for p in RAW_ROOT.rglob("*") if p.is_file()):
        relative = path.relative_to(RAW_ROOT)
        records.append(
            {
                "dataset": relative.parts[0] if relative.parts else "unknown",
                "path": str(relative),
                "suffix": "".join(path.suffixes).lower(),
                "bytes": path.stat().st_size,
                "size": human_bytes(path.stat().st_size),
            }
        )
inventory = pd.DataFrame(records)
display(
    inventory if not inventory.empty else pd.DataFrame({"status": ["No raw artifacts downloaded"]})
)
if not inventory.empty:
    display(
        inventory.groupby("dataset", as_index=False)["bytes"]
        .sum()
        .assign(size=lambda x: x.bytes.map(human_bytes))
    )

,dataset,path,suffix,bytes,size
0,.gitkeep,.gitkeep,,1,1.00 B


,dataset,bytes,size
0,.gitkeep,1,1.00 B


## 3. Memory-bounded flow-file sampling

The reader samples only the first `RAW_ROW_LIMIT` rows. First-row sampling can be biased by time; it is used here for structural EDA, not final metrics.

In [3]:
FLOW_SUFFIXES = {".csv", ".binetflow", ".txt"}


def read_flow_sample(path: Path, rows: int = RAW_ROW_LIMIT) -> pd.DataFrame:
    attempts = [
        {"sep": ",", "nrows": rows, "low_memory": False},
        {"sep": None, "engine": "python", "nrows": rows},
    ]
    error = None
    for kwargs in attempts:
        try:
            frame = pd.read_csv(path, **kwargs)
            if len(frame.columns) > 1:
                return frame
        except Exception as exc:
            error = exc
    raise RuntimeError(f"Could not parse {path}: {error}")


flow_files = (
    []
    if inventory.empty
    else [RAW_ROOT / value for value in inventory.loc[inventory.suffix.isin(FLOW_SUFFIXES), "path"]]
)
samples: dict[str, pd.DataFrame] = {}
sample_errors = []
for path in flow_files:
    key = str(path.relative_to(RAW_ROOT))
    try:
        samples[key] = read_flow_sample(path)
    except Exception as exc:
        sample_errors.append({"path": key, "error": repr(exc)})
pd.DataFrame(
    [
        {"artifact": key, "sample_rows": len(value), "columns": len(value.columns)}
        for key, value in samples.items()
    ]
)

""


In [4]:
if sample_errors:
    display(pd.DataFrame(sample_errors))
for name, frame in samples.items():
    print(f"\n{name}: {frame.shape}")
    display(frame.head(3))

## 4. Schema and canonical-feature coverage

Coverage means that a known source alias exists. It does not prove semantic/unit correctness; adapter tests and the feasibility audit must establish that.

In [5]:
from cyberworld.adapters.flow_csv import COMMON_ALIASES


def normalize_column(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


coverage_rows = []
for artifact, frame in samples.items():
    present = {normalize_column(column) for column in frame.columns}
    for canonical, aliases in COMMON_ALIASES.items():
        coverage_rows.append(
            {
                "artifact": artifact,
                "canonical_feature": canonical,
                "available": any(alias in present for alias in aliases),
            }
        )
coverage = pd.DataFrame(coverage_rows)
if not coverage.empty:
    matrix = coverage.pivot(index="canonical_feature", columns="artifact", values="available")
    display(matrix.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1))
else:
    display(pd.DataFrame({"status": ["No flow samples available"]}))

,status
0,No flow samples available


## 5. Missingness, cardinality, and numeric quality

Look for all-null fields, identifiers with suspiciously low cardinality, infinities, extreme ranges, and features that may encode the capture or label.

In [6]:
quality_tables = {}
for artifact, frame in samples.items():
    rows = []
    for column in frame.columns:
        numeric = pd.to_numeric(frame[column], errors="coerce")
        rows.append(
            {
                "column": column,
                "dtype": str(frame[column].dtype),
                "missing_pct": 100 * frame[column].isna().mean(),
                "unique": frame[column].nunique(dropna=True),
                "numeric_pct": 100 * numeric.notna().mean(),
                "infinite": int(np.isinf(numeric.dropna()).sum()),
                "min": numeric.min(),
                "median": numeric.median(),
                "max": numeric.max(),
            }
        )
    quality_tables[artifact] = pd.DataFrame(rows).sort_values(
        ["missing_pct", "unique"], ascending=[False, True]
    )
for artifact, table in quality_tables.items():
    print(artifact)
    display(table.head(30))

## 6. Source-label distributions

These are source labels, not automatically MITRE stages. Inspect imbalance before choosing loss weights or sampling.

In [7]:
LABEL_CANDIDATES = {"label", "attackcat", "activity", "stage"}
label_tables = []
for artifact, frame in samples.items():
    for column in frame.columns:
        if normalize_column(column) in LABEL_CANDIDATES:
            counts = frame[column].fillna("<missing>").astype(str).value_counts(dropna=False)
            for label, count in counts.items():
                label_tables.append(
                    {
                        "artifact": artifact,
                        "column": column,
                        "label": label,
                        "count": int(count),
                        "percent": 100 * count / len(frame),
                    }
                )
labels = pd.DataFrame(label_tables)
display(labels if not labels.empty else pd.DataFrame({"status": ["No label columns found"]}))

,status
0,No label columns found


In [8]:
if not labels.empty:
    for (artifact, column), group in labels.groupby(["artifact", "column"]):
        plot = group.sort_values("count").tail(20)
        ax = plot.plot.barh(
            x="label", y="count", figsize=(9, max(3, len(plot) * 0.28)), legend=False
        )
        ax.set_title(f"{artifact} — {column} (sample)")
        ax.set_xlabel("Rows")
        plt.tight_layout()
        plt.show()

## 7. Temporal coverage and density

Timestamp parsing failures and long gaps determine valid window size and trajectory boundaries.

In [9]:
TIME_CANDIDATES = {"timestamp", "starttime", "stime"}
temporal_rows = []
for artifact, frame in samples.items():
    candidate = next((c for c in frame.columns if normalize_column(c) in TIME_CANDIDATES), None)
    if candidate is None:
        temporal_rows.append({"artifact": artifact, "status": "no timestamp column"})
        continue
    raw = frame[candidate]
    numeric = pd.to_numeric(raw, errors="coerce")
    if numeric.notna().mean() > 0.95:
        unit = "ms" if numeric.dropna().median() > 10_000_000_000 else "s"
        parsed = pd.to_datetime(numeric, unit=unit, utc=True, errors="coerce")
    else:
        parsed = pd.to_datetime(raw, utc=True, errors="coerce", dayfirst=True, format="mixed")
    ordered = parsed.dropna().sort_values()
    gaps = ordered.diff().dt.total_seconds().dropna()
    temporal_rows.append(
        {
            "artifact": artifact,
            "column": candidate,
            "parse_success_pct": 100 * parsed.notna().mean(),
            "start": ordered.min() if not ordered.empty else None,
            "end": ordered.max() if not ordered.empty else None,
            "sample_span_hours": (ordered.max() - ordered.min()).total_seconds() / 3600
            if len(ordered)
            else 0,
            "median_gap_s": gaps.median() if not gaps.empty else None,
            "p99_gap_s": gaps.quantile(0.99) if not gaps.empty else None,
        }
    )
temporal = pd.DataFrame(temporal_rows)
display(temporal)

""


## 8. Representative PCAP inspection

This samples packet headers and transport payload sizes. It does not retain payload content. CTU's botnet-only PCAP must not be mistaken for complete mixed traffic.

In [10]:
pcap_files = (
    []
    if inventory.empty
    else [
        RAW_ROOT / value for value in inventory.loc[inventory.suffix.str.contains("pcap"), "path"]
    ]
)
pcap_rows = []
if pcap_files:
    from scapy.all import IP, TCP, UDP, PcapReader

    for path in pcap_files:
        protocols, ttl_values, windows, payload_sizes, flags = Counter(), [], [], [], Counter()
        read = 0
        with PcapReader(str(path)) as reader:
            for packet in reader:
                read += 1
                if IP in packet:
                    ttl_values.append(int(packet[IP].ttl))
                if TCP in packet:
                    protocols["TCP"] += 1
                    windows.append(int(packet[TCP].window))
                    payload_sizes.append(len(bytes(packet[TCP].payload)))
                    for flag in "SAFRPU":
                        if flag in str(packet[TCP].flags):
                            flags[flag] += 1
                elif UDP in packet:
                    protocols["UDP"] += 1
                    payload_sizes.append(len(bytes(packet[UDP].payload)))
                else:
                    protocols["OTHER"] += 1
                if read >= PCAP_PACKET_LIMIT:
                    break
        pcap_rows.append(
            {
                "artifact": str(path.relative_to(RAW_ROOT)),
                "sampled_packets": read,
                "protocols": dict(protocols),
                "ttl_mean": np.mean(ttl_values) if ttl_values else None,
                "ttl_std": np.std(ttl_values) if ttl_values else None,
                "tcp_window_median": np.median(windows) if windows else None,
                "payload_p95": np.quantile(payload_sizes, 0.95) if payload_sizes else None,
                "tcp_flags": dict(flags),
            }
        )
display(pd.DataFrame(pcap_rows) if pcap_rows else pd.DataFrame({"status": ["No PCAP available"]}))

,status
0,No PCAP available


## 9. Prepared trajectory overview

This is the model-facing representation. Each file must remain an independent scenario.

In [11]:
from cyberworld.labels import AttackStage
from cyberworld.preprocessing import STATE_FEATURES

trajectory_payloads = {}
trajectory_rows = []
for path in sorted(TRAJECTORY_ROOT.glob("*.npz")) if TRAJECTORY_ROOT.exists() else []:
    payload = dict(np.load(path, allow_pickle=False))
    trajectory_payloads[path.name] = payload
    states = payload["states"]
    availability = payload.get("availability", np.ones_like(states))
    stages = payload["stages"]
    trajectory_rows.append(
        {
            "trajectory": path.name,
            "windows": len(states),
            "features": states.shape[1],
            "feature_availability_pct": 100 * availability.mean(),
            "valid_label_pct": 100 * payload.get("label_valid", np.ones(len(states))).mean(),
            "stages_present": ", ".join(AttackStage(int(x)).name for x in np.unique(stages)),
        }
    )
trajectory_summary = pd.DataFrame(trajectory_rows)
display(
    trajectory_summary
    if not trajectory_summary.empty
    else pd.DataFrame({"status": ["No prepared trajectories"]})
)

,status
0,No prepared trajectories


## 10. State-feature availability and dynamics

Low availability identifies features that need PCAP extraction or a source-specific missingness strategy.

In [12]:
availability_rows = []
for name, payload in trajectory_payloads.items():
    mask = payload.get("availability", np.ones_like(payload["states"]))
    feature_names = [str(x) for x in payload.get("feature_names", np.asarray(STATE_FEATURES))]
    availability_rows.append(pd.Series(mask.mean(axis=0), index=feature_names, name=name))
if availability_rows:
    availability_matrix = pd.DataFrame(availability_rows)
    display(
        (100 * availability_matrix)
        .round(1)
        .style.background_gradient(cmap="RdYlGn", vmin=0, vmax=100)
    )
else:
    availability_matrix = pd.DataFrame()
    display(pd.DataFrame({"status": ["Prepare trajectories to inspect feature coverage"]}))

,status
0,Prepare trajectories to inspect feature coverage


In [13]:
for name, payload in trajectory_payloads.items():
    states = payload["states"]
    features = [str(x) for x in payload.get("feature_names", np.asarray(STATE_FEATURES))]
    selected = [
        x
        for x in ("total_flows", "total_packets", "total_bytes", "syn_rate", "dst_port_fanout")
        if x in features
    ]
    if not selected:
        continue
    indices = [features.index(x) for x in selected]
    view = pd.DataFrame(states[:, indices], columns=selected)
    view.plot(
        subplots=True,
        figsize=(12, 2.2 * len(selected)),
        title=f"Window states — {name}",
        legend=False,
    )
    plt.xlabel("Window index")
    plt.tight_layout()
    plt.show()

## 11. Operational stages and observed transitions

A transition count is not automatically an attack-progression claim. Only trajectories marked eligible by audited source timelines may be used for ordered-stage metrics.

In [14]:
stage_counts = Counter()
transition_counts = Counter()
for payload in trajectory_payloads.values():
    stages = payload["stages"].astype(int)
    valid = payload.get("label_valid", np.ones(len(stages))).astype(bool)
    for stage in stages[valid]:
        stage_counts[AttackStage(stage).name] += 1
    for left, right, keep in zip(stages[:-1], stages[1:], valid[:-1] & valid[1:], strict=True):
        if keep:
            transition_counts[(AttackStage(left).name, AttackStage(right).name)] += 1
display(
    pd.DataFrame([{"stage": k, "windows": v} for k, v in stage_counts.items()]).sort_values(
        "windows", ascending=False
    )
    if stage_counts
    else pd.DataFrame({"status": ["No valid stage windows"]})
)
transition_frame = pd.DataFrame(
    [{"from": k[0], "to": k[1], "count": v} for k, v in transition_counts.items()]
)
if not transition_frame.empty:
    transition_matrix = transition_frame.pivot_table(
        index="from", columns="to", values="count", fill_value=0
    )
    display(transition_matrix.style.background_gradient(cmap="Blues"))
else:
    display(pd.DataFrame({"status": ["No valid transitions"]}))

,status
0,No valid stage windows


,status
0,No valid transitions


## 12. Readiness summary

This is a local-data checklist, not a substitute for the signed-off feasibility audit.

In [15]:
datasets_present = set(inventory.dataset) if not inventory.empty else set()
checks = [
    ("At least two independent trajectories", len(trajectory_payloads) >= 2),
    ("At least one raw flow artifact", bool(samples)),
    ("At least one PCAP artifact", bool(pcap_files)),
    ("No sampled flow parse errors", not sample_errors),
    (
        "Canonical state dimension is 48",
        all(x["features"] == 48 for x in trajectory_rows) if trajectory_rows else False,
    ),
    (
        "At least one malicious operational label",
        any(name not in {"NORMAL", "UNKNOWN"} for name in stage_counts),
    ),
]
readiness = pd.DataFrame(checks, columns=["check", "passed"])
display(
    readiness.style.map(
        lambda x: (
            "background-color: #c6efce"
            if x is True
            else ("background-color: #ffc7ce" if x is False else "")
        ),
        subset=["passed"],
    )
)
print(f"Passed {int(readiness.passed.sum())}/{len(readiness)} local readiness checks")

,check,passed
0,At least two independent trajectories,False
1,At least one raw flow artifact,False
2,At least one PCAP artifact,False
3,No sampled flow parse errors,True
4,Canonical state dimension is 48,False
5,At least one malicious operational label,False


Passed 1/6 local readiness checks


## Interpretation prompts

Before training, answer these from the outputs above:

1. Are attack labels sparse enough that accuracy would be misleading?
2. Which canonical features are shared, and which require packet extraction?
3. Are timestamp gaps compatible with 5 s, 10 s, or 30 s windows?
4. Do apparent stage transitions come from documented campaigns or merely adjacent attack categories?
5. Could IPs, dates, filenames, or ports leak the dataset label?
6. Is external-test feature availability lower than training availability?
7. Which claims must be narrowed because the required labels are absent?